# Wireless Coverage — Shared Configuration

Shared config, imports, and helper functions for the San Francisco LiDAR wireless-coverage
series (`01_pure_lidar_chm` through `04_tower_viewsheds`). Imported by each notebook via
`%run ./config_nb`; installs **no packages** — the calling notebook owns `%pip` and
`%restart_python`. Registers GeoBrix UDFs and defines series-wide helpers.

GeoBrix functions in scope after `%run`:
- `rx.*` — pyrx columnar raster functions (Serverless-safe, no JAR)
- `gx.*` — pygx columnar grid functions (H3, BNG)
- `register(spark)` — installs `gbx_*` SQL UDFs (readers + pyrx + pygx)
- Vizx helpers: `plot_raster`, `plot_file`, `plot_static`, `plot_point_cloud`,
  `show_flythrough`, `raster_layer`, `point_cloud_layer`, `vector_layer`, `cells_as_gdf`

Runtime: **Databricks Serverless environment 6** — lightweight tier (pure Python/PySpark, no JAR).

---
**Last Update:** 2026-10-03

In [0]:
# ============================================================================
# Wireless Coverage — imports (shared across nb1–nb4)
# ============================================================================
import warnings
warnings.filterwarnings("ignore")

import glob, math, time

import numpy as np
import pandas as pd

from pyspark.sql import functions as F, types as T, Window
from pyspark.databricks.sql import functions as DBF

from shapely import wkb as _wkb
from shapely.geometry import box as _box
from shapely.ops import unary_union

# GeoBrix lightweight tier (Serverless-safe, no JAR)
from databricks.labs.gbx.pyrx import functions as rx
from databricks.labs.gbx.pygx import functions as gx
from databricks.labs.gbx.pygx import _cellfill
from databricks.labs.gbx.ds.register import register
from databricks.labs.gbx.sample.lidar import LidarDownloader, aoi_lonlat_to_3857, lonlat_to_3857
from databricks.labs.gbx.sample.overture import OvertureClient
from databricks.labs.gbx.vizx import (
    plot_raster, plot_file, plot_static, plot_point_cloud, show_flythrough,
    raster_layer, point_cloud_layer, vector_layer, cells_as_gdf,
)

In [0]:
rx.register(spark)
gx.register(spark)
register(spark)   # installs lidar_gbx reader + gbx_rst_* / gbx_bng_* / gbx_st_* SQL UDFs

In [0]:
# ── Series-wide parameters ─────────────────────────────────────────────────
# SF AOI (lon/lat, EPSG:4326). Matches across nb1–nb4.
SF_AOI  = (-122.55, 37.70, -122.35, 37.85)

# Staged USGS 3DEP LiDAR (.laz EPT nodes, EPSG:3857).
LAZ_DIR = "/Volumes/geospatial_docs/wireless_coverage/data/lidar/sf/laz"

# Path-backed LiDAR surfaces (DSM/DTM/CHM GeoTIFF tiles, EPSG:3857) — written as gtiff_gbx
# dirs {OUT_DIR}/{dsm,dtm,chm} by nb1 or nb2, and consumed by nb4's viewshed windowing.
# Lives beside the staged LAZ under the wireless_coverage tree.
OUT_DIR = "/Volumes/geospatial_docs/wireless_coverage/data/lidar/sf/lidar-surfaces"

# Unity Catalog target (must be writable).
CATALOG = "geospatial_docs"
SCHEMA  = "geobrix"

# Raster grid: 1 m pixels, 1024 m tiles (1024 × 1024 px), EPSG:3857.
SRID    = 3857
PIXEL_M = 1.0
TILE_M  = 1024.0
TPX     = int(TILE_M / PIXEL_M)

# H3 default resolution for the series (res 10 ≈ 65 m edge length).
# Calling notebooks may override this before any _persist / _surface_to_h3_cells call.
H3_RESOLUTION = 10

# ── Runtime flags (safe defaults; calling notebooks override as needed) ────
FORCE_RELOAD   = False   # True → rebuild output tables even if they exist
FORCE_DOWNLOAD = False   # True → re-fetch LiDAR / Overture data
DEMO           = False   # True → restrict to Golden Gate Park ±2 tile window
INTERACTIVE    = False   # True → live deck.gl renders (Databricks sessions only)

# ── AOI in EPSG:3857 (anchors the tile grid) ──────────────────────────────
X0, Y0, X1, Y1 = aoi_lonlat_to_3857(*SF_AOI)
print(f"AOI EPSG:3857: x=[{X0:.0f}, {X1:.0f}] y=[{Y0:.0f}, {Y1:.0f}]")
print(f"Tile: {TILE_M:.0f} m = {TPX} px | H3_RESOLUTION = {H3_RESOLUTION}")

In [0]:
spark.catalog.setCurrentCatalog(CATALOG)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.catalog.setCurrentDatabase(SCHEMA)

In [0]:
def _tbl(prefix, name, res=None):
    """Build a fully-qualified table name with demo and resolution markers.

    ``_demo`` suffix appended when DEMO=True, so demo-subset tables never
    clobber or confuse a full-SF run. ``_res{res}`` appended when res is given
    (H3 output tables only — not raster surface tables).

    Examples::

        # Full SF, raster surface:  wc_surface_dsm
        # Full SF, H3 res 10:       wc_h3_dsm_res10
        # DEMO SF, raster surface:  wc_surface_dsm_demo
        # DEMO SF, H3 res 10:       wc_h3_dsm_demo_res10
    """
    s = f"{prefix}_{name}"
    if DEMO:
        s += "_demo"
    if res is not None:
        s += f"_res{res}"
    return f"{CATALOG}.{SCHEMA}.{s}"

In [0]:
def _persist(df, name, prefix=None, res=None):
    """Write df to ``CATALOG.SCHEMA.{prefix}_{name}[_res{res}]`` and return it.

    ``prefix`` defaults to the first of ``TABLE_PREFIX``, ``OUT_PREFIX``, or ``"wc"``
    found in the caller's global namespace — so each notebook sets its own prefix
    after ``%run ./config_nb`` and _persist picks it up automatically.

    Pass ``res=H3_RESOLUTION`` for H3 output tables to tag the resolution in the
    table name (e.g. ``wc_h3_dsm_res10``).  Omit ``res`` for raster surface tables
    (e.g. ``wc_surface_dsm``).

    ``FORCE_RELOAD=True`` rebuilds; otherwise an existing table is reused.
    """
    if prefix is None:
        prefix = globals().get("TABLE_PREFIX") or globals().get("OUT_PREFIX") or "wc"
    tbl = _tbl(prefix, name, res)
    if FORCE_RELOAD or not spark.catalog.tableExists(tbl):
        df.write.mode("overwrite").saveAsTable(tbl)
    return spark.table(tbl)

In [0]:
def preview(df, label=None, n=10, order_by=None, vertical=None, count=True):
    """GitHub-safe DataFrame preview: count as narrative + bounded display.

    ``count=True`` (default): prints ``df.count()`` as narrative — cheap on a
    table-backed DataFrame, but triggers full recomputation on a lazy/ephemeral one.
    Pass ``count=False`` for un-persisted intermediates, or persist first.

    In a live Databricks notebook, `display()` renders an interactive table; on
    GitHub, the committed output is static — so we always `.limit(n)`. For rows
    with binary / struct columns (raster tiles, WKB), a single-row vertical show
    is more readable than truncated columns.

    Args:
        df:       Spark DataFrame to preview.
        label:    Optional narrative label printed after the row count.
        n:        Max rows to display (default 10).
        order_by: Column name(s) to sort by before limiting (default None).
        vertical: Force vertical show (True) or wide (False). Auto-detects
                  BinaryType / StructType columns when None.
    """
    if count:
        _n = df.count()
        print(f"{_n:,} {label}" if label else f"count: {_n:,}")
    df2 = df.orderBy(order_by) if order_by is not None else df
    heavy = (
        vertical
        if vertical is not None
        else any(
            isinstance(f.dataType, (T.BinaryType, T.StructType))
            for f in df.schema.fields
        )
    )
    if heavy:
        df2.limit(1).show(vertical=True)
    else:
        try:
            display(df2.limit(n))   # rich in Databricks; bounded for GitHub
        except NameError:
            df2.limit(n).show()     # fallback outside a notebook


def _points_as_gdf(df, lon_col="lon", lat_col="lat", value_cols=(), max_points=50_000):
    """Spark DataFrame of lon/lat rows -> GeoDataFrame of Points (EPSG:4326).

    Collects to the driver (safe for the small point sets rendered as markers --
    tower sites, sample locations); carries value_cols through for colouring.
    """
    import geopandas as _gpd
    from shapely.geometry import Point as _Point

    pdf = df.select(lon_col, lat_col, *value_cols).limit(max_points).toPandas()
    geom = [_Point(float(x), float(y)) for x, y in zip(pdf[lon_col], pdf[lat_col])]
    return _gpd.GeoDataFrame(
        {c: pdf[c] for c in value_cols}, geometry=geom, crs="EPSG:4326"
    )

In [0]:
def _surface_to_h3_cells(surface_df, tile_col, breaks_array, h3_res, land_wkb=None):
    """EPSG:3857 surface tiles → elevation isobands → H3 cells (EPSG:4326).

    Steps:
      1. rst_transform(tile, 4326)        — REQUIRED: h3_try_coverash3 needs lon/lat WKB.
      2. rst_clip(land_wkb) [optional]    — mask Bay/Pacific to NoData before isoband.
      3. rst_isoband(breaks_array)         — produces (band_level, geom_wkb, elev_lo, elev_hi).
      4. h3_try_coverash3(geom_wkb, h3_res) + explode — solid, gap-free H3 cells per band.

    Args:
        surface_df:   DataFrame with (tx INT, ty INT, <tile_col> STRUCT) — EPSG:3857 tiles.
        tile_col:     Column name of the raster tile (e.g. 'dtm', 'dsm').
        breaks_array: Spark Column of type ARRAY<DOUBLE> — elevation break thresholds.
        h3_res:       H3 resolution integer (e.g. 10).
        land_wkb:     Optional bytes — EPSG:4326 land polygon WKB cutline for rst_clip.
                      Pass None to skip the land-mask step.

    Returns:
        DataFrame with columns (cellid LONG, res INT, band_level INT, elev_lo DOUBLE,
        elev_hi DOUBLE), one row per (H3 cell, elevation band).
    """
    # Fine-grained repartition: one tile per task keeps Python UDF worker memory
    # bounded (avoids UDF_PYSPARK_ERROR.OOM at full-SF scale on Serverless).
    surface_df = surface_df.repartition(512, "tx", "ty")
    # Step 1: reproject 3857 → 4326 (lon/lat required for H3 product functions).
    t4326 = surface_df.select(
        "tx", "ty",
        rx.rst_transform(tile_col, F.lit(4326)).alias("tile_4326"),
    )
    # Step 2: clip to land polygon (removes Bay/Pacific pixels before isoband).
    if land_wkb is not None:
        clipped = t4326.select(
            "tx", "ty",
            rx.rst_clip("tile_4326", F.lit(land_wkb), F.lit(True)).alias("tile"),
        )
    else:
        clipped = t4326.withColumnRenamed("tile_4326", "tile")
    # Step 3: extract contiguous elevation-band polygons.
    patches = (
        clipped
        .select(F.explode(rx.rst_isoband("tile", breaks_array)).alias("p"))
        .select(
            F.col("p.band").alias("band_level"),
            F.col("p.geom_wkb").alias("geom_wkb"),
            F.col("p.lower").alias("elev_lo"),
            F.col("p.upper").alias("elev_hi"),
        )
    )
    # Step 4: H3 index via Databricks product h3_try_coverash3 (WKB input, no ST_ wrap).
    cells = (
        patches
        .select(
            "band_level", "elev_lo", "elev_hi",
            F.explode(
                DBF.h3_try_coverash3(F.col("geom_wkb"), F.lit(h3_res))
            ).alias("cellid"),
        )
        .where(F.col("cellid").isNotNull())
        .distinct()
        .withColumn("res", F.lit(h3_res))
    )
    return cells.select("cellid", "res", "band_level", "elev_lo", "elev_hi")

In [0]:
def _cellfill_depth(cells_df, value_col, k=1):
    """Fill NULL-valued gap cells (k-ring IDW) from valid H3 neighbours.

    cells_df must have (cellid LONG, <value_col> DOUBLE) with unique cellid per row.
    Returns a DataFrame with the same schema including filled cells.

    Args:
        cells_df:  DataFrame with (cellid LONG, <value_col> DOUBLE).
        value_col: Column name of the value to interpolate.
        k:         Ring size for gap detection (default 1 — immediate neighbours only).

    # GeoBrix-promotion candidate → pygx.h3_tools (public)
    """
    gap = (
        cells_df
        .select(F.explode(DBF.h3_kring(F.col("cellid"), F.lit(k))).alias("cellid"))
        .distinct()
        .join(cells_df.select("cellid"), on="cellid", how="left_anti")
        .withColumn(value_col, F.lit(None).cast("double"))
    )
    fill_in = cells_df.select("cellid", value_col).unionByName(gap).withColumn("_g", F.lit(1))
    payload = fill_in.groupBy("_g").agg(
        gx.h3_cellfill("cellid", value_col, F.lit(k), F.lit("idw"), F.lit(2.0)).alias("p")
    ).first()["p"]
    rows = [(int(cid), v) for cid, v in _cellfill.decode(payload) if v is not None]
    return spark.createDataFrame(rows, f"cellid long, {value_col} double")

In [0]:
def _tile_range(coord_3857, origin_3857, radius, tile_size):
    """Integer tx/ty range for a 1D window centred on coord_3857.

    Returns (tx_min, tx_max) inclusive — tile indices whose extent intersects
    [coord - radius, coord + radius].

    # GeoBrix-promotion candidate → ds.tiles / pyrx tile arithmetic
    """
    tx_min = math.floor((coord_3857 - origin_3857 - radius) / tile_size)
    tx_max = math.floor((coord_3857 - origin_3857 + radius) / tile_size)
    return int(tx_min), int(tx_max)


# ── Unit tests (run at import time via %run) ───────────────────────────────
assert _tile_range(1000, 0, 150, 100) == (8, 11), "_tile_range test 1 failed"
assert _tile_range(1024, 0, 5000, 1024) == (-4, 5), "_tile_range test 2 failed"
assert _tile_range(512,  0, 1024, 1024) == (-1, 1), "_tile_range test 3 failed"
print("_tile_range unit tests PASSED")

In [0]:
def _window_tiles(tower_df, dsm_df, max_dist_m, x0, y0, tile_m, n_partitions=None):
    """Range-join towers to DSM tiles and merge into one tile per tower.

    Args:
        tower_df:   DataFrame with (tower_cellid, tower_x, tower_y, center_wkb).
        dsm_df:     DataFrame with (tx INT, ty INT, dsm STRUCT) — EPSG:3857 tiles.
        max_dist_m: Viewshed radius (metres).
        x0, y0:     EPSG:3857 AOI origin (X0, Y0 from aoi_lonlat_to_3857).
        tile_m:     Tile size in metres (TILE_M parameter).
        n_partitions: Fan-out for the merge — repartition the join by tower_cellid into
            this many partitions BEFORE the groupBy so AQE cannot coalesce many towers'
            tile bytes onto one task. Defaults to the distinct tower count (~1 tower/task).

    Returns:
        DataFrame with (tower_cellid, center_wkb, tower_x, tower_y, dsm_window) —
        one merged tile per tower.

    Notes:
        - Uses rst_merge_agg (group aggregator) — safe for FILE tiles on Spark 4.0.
        - Arithmetic range join: no ST_ geometry dependency.

    # GeoBrix-promotion candidate → pyrx raster windowing
    """
    R = max_dist_m
    towers_ranged = tower_df.select(
        "tower_cellid",
        "tower_x", "tower_y",
        "center_wkb",
        F.floor((F.col("tower_x") - x0 - R) / tile_m).cast("int").alias("tx_min"),
        F.floor((F.col("tower_x") - x0 + R) / tile_m).cast("int").alias("tx_max"),
        F.floor((F.col("tower_y") - y0 - R) / tile_m).cast("int").alias("ty_min"),
        F.floor((F.col("tower_y") - y0 + R) / tile_m).cast("int").alias("ty_max"),
    )
    joined = towers_ranged.join(
        dsm_df,
        (
            (F.col("tx") >= F.col("tx_min")) &
            (F.col("tx") <= F.col("tx_max")) &
            (F.col("ty") >= F.col("ty_min")) &
            (F.col("ty") <= F.col("ty_max"))
        ),
        "inner",
    )
    # Fan out BEFORE the aggregation: an explicit repartition by tower_cellid is a
    # re-partition exchange AQE cannot coalesce past (the only mechanism that survives
    # Serverless), so each merge task holds the tiles of a bounded set of towers rather
    # than AQE piling many towers' tile bytes onto one worker (OOM).
    if n_partitions is None:
        n_partitions = max(tower_df.select("tower_cellid").distinct().count(), 1)
    # One merged DSM tile per tower — rst_merge_agg is safe for FILE tiles on Spark 4.
    windowed = (
        joined.repartition(n_partitions, "tower_cellid")
        .groupBy("tower_cellid", "center_wkb", "tower_x", "tower_y")
        .agg(rx.rst_merge_agg("dsm").alias("dsm_window"))
    )
    return windowed


def _window_vrt(tower_df, dsm_tiles_df, max_dist_m, x0, y0, tile_m, vrt_dir, n_partitions=None):
    """Per-tower virtual DEM window: range-join towers to the in-radius DSM tile paths,
    then mint ONE VRT per tower over those paths. The VRT extent is the window, so the
    downstream viewshed read is bounded without pixel-window math and NO raster bytes
    cross the shuffle — each row carries a ~100-byte VRT path.

    Args:
        tower_df:     DataFrame with (tower_cellid, center_wkb, observer_height, tower_x, tower_y).
        dsm_tiles_df: DataFrame with (tx INT, ty INT, tile_path STRING) — the FILE-column DSM
                      tile paths (project dsm.path/tile.path to tile_path before calling).
        max_dist_m:   Viewshed radius (metres); defines the in-radius tile window.
        x0, y0:       EPSG:3857 AOI origin (X0, Y0).
        tile_m:       Tile size in metres (TILE_M).
        vrt_dir:      Durable, worker-readable dir (UC Volume) for the per-tower VRTs.
        n_partitions: Optional fan-out before minting (repartition by tower_cellid).

    Returns:
        DataFrame (tower_cellid, center_wkb, observer_height, tower_x, tower_y, dem_vrt_path) —
        one bytes-free per-tower VRT path.

    Notes:
        - mint_vrt runs in a worker UDF (parallel header reads + small VRT write); it is
          Connect-safe (pure Python + rasterio, no osgeo / no _jvm). If worker Volume writes
          prove flaky, mint on the driver from collected path lists.
        - VRT members must share CRS / pixel size / dtype / bands (a native-grid DSM does).
    """
    R = max_dist_m
    towers_ranged = tower_df.select(
        "tower_cellid", "center_wkb", "observer_height", "tower_x", "tower_y",
        F.floor((F.col("tower_x") - x0 - R) / tile_m).cast("int").alias("tx_min"),
        F.floor((F.col("tower_x") - x0 + R) / tile_m).cast("int").alias("tx_max"),
        F.floor((F.col("tower_y") - y0 - R) / tile_m).cast("int").alias("ty_min"),
        F.floor((F.col("tower_y") - y0 + R) / tile_m).cast("int").alias("ty_max"),
    )
    joined = towers_ranged.join(
        dsm_tiles_df.select("tx", "ty", "tile_path"),
        (F.col("tx") >= F.col("tx_min")) & (F.col("tx") <= F.col("tx_max"))
        & (F.col("ty") >= F.col("ty_min")) & (F.col("ty") <= F.col("ty_max")),
        "inner",
    )
    per_tower = joined.groupBy(
        "tower_cellid", "center_wkb", "observer_height", "tower_x", "tower_y"
    ).agg(F.collect_list("tile_path").alias("tile_paths"))
    if n_partitions:
        per_tower = per_tower.repartition(n_partitions, "tower_cellid")

    @F.udf("string")
    def _mint_tower_vrt(cellid, paths):
        import os
        from databricks.labs.gbx.ds._mosaic import mint_vrt
        if not paths:
            return None
        out = os.path.join(vrt_dir, "tower_{}.vrt".format(cellid))
        return mint_vrt(sorted(set(paths)), out=out)

    return (
        per_tower
        .withColumn("dem_vrt_path", _mint_tower_vrt("tower_cellid", "tile_paths"))
        .drop("tile_paths")
    )


In [0]:
# ── lon/lat → EPSG:3857 UDF ────────────────────────────────────────────────
# Pure-math Mercator; avoids st_transform Photon dependency.
# Mirrors databricks.labs.gbx.sample.lidar.lonlat_to_3857 exactly.
_MERC_R = 20037508.342789244

from pyspark.sql.types import StructType, StructField, DoubleType as _DT

@F.udf(StructType([StructField("x", _DT()), StructField("y", _DT())]))
def _lonlat_to_3857(lon, lat):
    """Spherical Mercator (EPSG:3857): same formula as sample.lidar.lonlat_to_3857."""
    import math as _math
    x = lon * _MERC_R / 180.0
    lat_r = _math.radians(lat)
    y = _math.log(_math.tan(lat_r / 2.0 + _math.pi / 4.0)) * (_MERC_R / _math.pi)
    return {"x": x, "y": y}

In [0]:
# ── H3 center → (lat, lon, wkb) fallback UDF ───────────────────────────────
# Used when DBF.h3_centeraswkb is unavailable on the current compute environment.
# Notebooks set _CENTER_FN via a T0 probe and call this UDF only as the fallback.
from pyspark.sql.types import StructType, StructField, DoubleType, BinaryType

@F.udf(StructType([
    StructField("lat", DoubleType()),
    StructField("lon", DoubleType()),
    StructField("wkb", BinaryType()),
]))
def _h3_center(cellid):
    """H3 cell → (lat, lon, WKB POINT) — fallback when h3_centeraswkb unavailable."""
    import h3 as _h3lib
    from shapely.geometry import Point
    from shapely import wkb as _wkb2
    lat, lng = _h3lib.cell_to_latlng(cellid)
    return {"lat": lat, "lon": lng, "wkb": _wkb2.dumps(Point(lng, lat))}